# Course 10 · Applied Data Science Capstone: SpaceX Falcon 9 landing prediction
**Business question:** can we predict whether the Falcon 9 first stage lands? A landing makes a launch much cheaper.
This notebook condenses the capstone labs into one runnable pipeline. *Needs internet (public SpaceX API). Every line has a `#` comment.*

## Lab 1 · Data collection with the SpaceX REST API

In [ ]:
import pandas as pd, numpy as np, requests

# Download all past launches as JSON. json_normalize flattens nested JSON into table columns.
url = "https://api.spacexdata.com/v4/launches/past"
launches = pd.json_normalize(requests.get(url, timeout=30).json())
print(launches.shape)                                       # (launches, columns)
launches[["name", "date_utc", "rocket", "success"]].head()  # "rocket" is only an ID code for now

### Turn ID codes into real values using lookup tables (as in the lab)

In [ ]:
def lookup(endpoint):
    """Download one API endpoint and return a dict: id -> full record."""
    records = requests.get(f"https://api.spacexdata.com/v4/{endpoint}", timeout=30).json()
    return {d["id"]: d for d in records}          # dict comprehension keyed by id

rockets, pads, payloads = lookup("rockets"), lookup("launchpads"), lookup("payloads")

# Keep only the columns we need.
df = launches[["flight_number", "date_utc", "rocket", "launchpad", "payloads", "cores"]].copy()
# Keep launches with exactly one core (booster) and at least one payload: the lab's simplification.
df = df[(df.cores.map(len) == 1) & (df.payloads.map(len) >= 1)]

# .map(function) applies the function to every value in the column.
df["BoosterVersion"] = df.rocket.map(lambda r: rockets[r]["name"])   # rocket ID -> rocket name
df = df[df.BoosterVersion == "Falcon 9"].copy()                     # Falcon 9 only

df["LaunchSite"] = df.launchpad.map(lambda p: pads[p]["name"])      # launchpad ID -> name
df["Latitude"]   = df.launchpad.map(lambda p: pads[p]["latitude"])
df["Longitude"]  = df.launchpad.map(lambda p: pads[p]["longitude"])

first_payload = df.payloads.map(lambda p: payloads[p[0]])           # record of the first payload
df["PayloadMass"] = first_payload.map(lambda p: p.get("mass_kg"))   # .get -> None if missing
df["Orbit"]       = first_payload.map(lambda p: p.get("orbit"))

core = df.cores.map(lambda c: c[0])                                 # the single core's details
for col in ["landing_success", "landing_type", "flight", "gridfins", "reused", "legs"]:
    df[col] = core.map(lambda c: c.get(col))                        # copy each detail into a column

df["Date"] = pd.to_datetime(df.date_utc).dt.date                    # timestamp -> plain date
# Drop the raw ID/list columns and renumber rows 0, 1, 2...
df = df.drop(columns=["rocket", "launchpad", "payloads", "cores", "date_utc"]).reset_index(drop=True)
df.head()

## Lab 3 · Data wrangling: missing values and the target label
`Class` = 1 if the booster landed, else 0.

In [ ]:
missing = df.isna().sum()
print(missing[missing > 0])                                        # which columns have gaps
df["PayloadMass"] = df["PayloadMass"].fillna(df["PayloadMass"].mean())   # fill with the average mass
df["flight"] = df["flight"].fillna(1)                              # unknown booster flight count -> 1st flight
df["Class"] = (df["landing_success"] == True).astype(int)          # True -> 1; False or None -> 0
df["Year"] = pd.to_datetime(df["Date"]).dt.year                    # year, for trend charts
print("Landing success rate:", df["Class"].mean().round(3))        # the mean of 0/1 = share of 1s
df[["flight_number", "LaunchSite", "Orbit", "PayloadMass", "Class"]].head()

## Lab 4 · EDA with SQL

In [ ]:
import sqlite3
con = sqlite3.connect("spacex.db")                                 # database file
# SQLite cannot store Python date objects, so convert Date to text first.
table = df.drop(columns=["landing_success"]).astype({"Date": str})
table.to_sql("SPACEXTBL", con, if_exists="replace", index=False)
sql = lambda s: pd.read_sql_query(s, con)                          # short helper: SQL -> DataFrame

display(sql("SELECT DISTINCT LaunchSite FROM SPACEXTBL"))          # the launch sites
display(sql("""SELECT LaunchSite, COUNT(*) AS n, ROUND(AVG(Class), 2) AS success
               FROM SPACEXTBL GROUP BY LaunchSite"""))             # launches and success rate per site
display(sql("SELECT MIN(Date) AS first_success FROM SPACEXTBL WHERE Class = 1"))   # first landing
display(sql("""SELECT Orbit, ROUND(AVG(PayloadMass)) AS avg_kg, ROUND(AVG(Class), 2) AS success
               FROM SPACEXTBL GROUP BY Orbit ORDER BY success DESC"""))

## Lab 5 · EDA with visualization

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns

fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
# Each dot is a launch; colour shows landed (1) or not (0). Later flights land more often.
sns.scatterplot(data=df, x="flight_number", y="LaunchSite", hue="Class", ax=ax[0])
sns.scatterplot(data=df, x="PayloadMass", y="Orbit", hue="Class", ax=ax[1])
# Success rate per year: group by year, average the 0/1 Class column.
df.groupby("Year")["Class"].mean().plot(marker="o", ax=ax[2], title="Success rate by year")
plt.tight_layout()
plt.show()

## Lab 6 · Folium map of launch sites

In [ ]:
import folium
from folium.plugins import MarkerCluster

m = folium.Map(location=[29.0, -95.0], zoom_start=4)   # centred on the southern USA
cl = MarkerCluster().add_to(m)                        # cluster markers at the same site
for _, r in df.iterrows():                            # one marker per launch
    folium.Marker([r.Latitude, r.Longitude],
                  icon=folium.Icon(color="green" if r.Class else "red"),   # green = landed
                  popup=f"{r.LaunchSite} · flight {r.flight_number}").add_to(cl)
m

## Lab 8 · Predictive analysis: four models with GridSearchCV (cv=10)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

feat_num = ["flight_number", "PayloadMass", "flight"]            # numeric features
feat_cat = ["Orbit", "LaunchSite", "gridfins", "reused", "legs"] # categorical features
# Convert categorical columns to text so True/False/None are all treated as categories.
X = df[feat_num + feat_cat].astype({c: str for c in feat_cat})
y = df["Class"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=2, stratify=y)

prep = ColumnTransformer([("n", StandardScaler(), feat_num),                       # scale numbers
                          ("c", OneHotEncoder(handle_unknown="ignore"), feat_cat)]) # one-hot categories

# name -> (model, settings to try). "clf__" points at the pipeline step named "clf".
grids = {
    "LogReg": (LogisticRegression(max_iter=1000), {"clf__C": [0.01, 0.1, 1]}),
    "SVM":    (SVC(), {"clf__kernel": ["linear", "rbf", "sigmoid"], "clf__C": [0.1, 1, 10]}),
    "Tree":   (DecisionTreeClassifier(random_state=0),
               {"clf__max_depth": [2, 4, 8, None], "clf__criterion": ["gini", "entropy"]}),
    "KNN":    (KNeighborsClassifier(), {"clf__n_neighbors": list(range(1, 11))}),
}
rows = []
for name, (clf, g) in grids.items():
    # 10-fold cross-validation for every setting, as in the capstone.
    gs = GridSearchCV(Pipeline([("prep", prep), ("clf", clf)]), g, cv=10).fit(X_tr, y_tr)
    rows.append({"model": name,
                 "cv_accuracy": gs.best_score_,              # average accuracy across the 10 folds
                 "test_accuracy": gs.score(X_te, y_te),      # accuracy on held-out launches
                 "best_params": gs.best_params_})            # the winning settings
pd.DataFrame(rows).set_index("model").round(3)

**Note:** the test set has only around 20 launches, so models often tie. Trust the 10-fold CV accuracy more than the test accuracy.

## Lab 9 · Presentation outline
1. Executive summary 2. Introduction 3. Methodology 4. EDA results (SQL, charts, map, dashboard) 5. Model results 6. Conclusion 7. Appendix

## ✍️ Your turn
1. Build the Dash app from the lab: launch-site dropdown, success pie chart, payload range slider (use notebook 08 Step 7 as a template).
2. Write a 3-bullet executive summary of what you found.

In [ ]:
# your answer here